# Tornado data: starter notebook

Run the setup and inspection cells before your investigation. After brainstorming with your group, use the open cells at the end for your own EDA.

**Dataset:** NOAA SPC historical tornado records, **2015-2018**. There are 4,706 whole-track records and 17 columns. One row represents one tornado track after removing duplicate and continuation records.

[Data guide](../data_dictionary.qmd) and [source notes](../sources.qmd). The same information is also summarized below.


## 1. Load the local CSV

Dates become datetimes. The EF rating uses pandas nullable integers so unknown ratings stay missing while EF0 stays zero.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data/tornadoes.csv").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data/tornadoes.csv").is_file():
    raise FileNotFoundError("Open the project folder, then run this notebook again.")

tornadoes = pd.read_csv(
    ROOT / "data/tornadoes.csv",
    parse_dates=["event_date"],
    dtype={"tornado_id": "string", "start_state": "string", "ef_rating": "Int64"},
)
sns.set_theme(style="whitegrid", context="talk")
print(f"Loaded {len(tornadoes):,} records and {tornadoes.shape[1]} columns.")


Loaded 4,706 records and 17 columns.


## 2. Preview a few records

Read the units in the column names. A whole track can begin and end at different coordinates.


In [2]:
tornadoes[[
    "tornado_id", "event_date", "start_state", "ef_rating",
    "path_length_miles", "max_width_yards", "states_affected"
]].head()


,tornado_id,event_date,start_state,ef_rating,path_length_miles,max_width_yards,states_affected
0,2015-549199,2015-01-03,MS,0,0.08,50.0,1
1,2015-549208,2015-01-03,MS,1,1.88,50.0,1
2,2015-549220,2015-01-03,MS,1,1.03,100.0,1
3,2015-549233,2015-01-03,MS,2,8.03,200.0,1
4,2015-549251,2015-01-03,MS,1,9.89,100.0,1


## 3. Understand the fields

| Kind of information | Columns | Meaning |
|---|---|---|
| Identifier | `tornado_id` | A year-event identifier, not a measurement |
| Date | `event_date`, `year`, `month` | Source start date and derived calendar fields |
| Clock time | `start_time_cst`, `start_hour_cst` | CST for every record, not local time everywhere |
| Starting location | `start_state`, `start_latitude`, `start_longitude` | Where the recorded track begins |
| Ending location | `end_latitude`, `end_longitude` | Where the recorded track ends |
| Track size | `path_length_miles`, `max_width_yards` | Total length and maximum width in different units |
| Intensity | `ef_rating` | Ordered EF damage category, not a direct wind-speed measurement |
| Extent and outcomes | `states_affected`, `injuries`, `fatalities` | Recorded counts associated with the whole track |

EF0 is valid. An unknown EF rating is `<NA>`. The categories are ordered, but EF4 does not mean twice the wind speed of EF2. The rating is based on damage surveys and estimated winds.

A multi-state track still occupies one row. `start_state` assigns it to its beginning, not every state it affected. All start hours use a common CST clock, including records outside that zone.

The snapshot has no city or county name, narrative description, end time, duration, population, or exposed-area denominator.


In [3]:
tornadoes.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4706 entries, 0 to 4705
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   tornado_id         4706 non-null   string        
 1   event_date         4706 non-null   datetime64[ns]
 2   year               4706 non-null   int64         
 3   month              4706 non-null   int64         
 4   start_time_cst     4706 non-null   object        
 5   start_hour_cst     4706 non-null   int64         
 6   start_state        4706 non-null   string        
 7   ef_rating          4595 non-null   Int64         
 8   path_length_miles  4706 non-null   float64       
 9   max_width_yards    4706 non-null   float64       
 10  start_latitude     4706 non-null   float64       
 11  start_longitude    4706 non-null   float64       
 12  end_latitude       4706 non-null   float64       
 13  end_longitude      4706 non-null   float64       
 14  states_a

## 4. Check missing information

This checks availability, without choosing your research question or dropping records.


In [4]:
missing = tornadoes.isna().sum()
display(missing[missing.gt(0)].rename("missing_records").to_frame())
print("Unknown EF ratings:", tornadoes["ef_rating"].isna().sum())
print("Records with an observed EF rating:", tornadoes["ef_rating"].notna().sum())


Unknown EF ratings: 111
Records with an observed EF rating: 4595


,missing_records
ef_rating,111


Keep the full DataFrame. If your chosen question requires EF rating, decide explicitly how to handle the 111 unknown values. A question using dates or path lengths may not need to exclude them.

`.dropna()` with no subset would drop records for every missing field. Prefer `dropna(subset=[...])` for the variables your question actually needs, and report how many rows remain.

## 5. Your investigation

Choose a question after your group's brainstorming. You can use ordinary pandas, seaborn, or matplotlib code below.

**Our question:** ...

**Variables, units, comparison, and any denominator:** ...


In [5]:
# Your first summary or visualization.


**What does the result show? What remains unresolved?** ...

**Our follow-up question or check:** ...


In [6]:
# Your follow-up summary or visualization.


## Before regrouping

Keep your plot available. Be ready to explain the question, evidence, how your investigation changed, and what you would investigate next.

If you need syntax support, see `plotting_reference.ipynb`. Its examples use Titanic so your tornado investigation remains your own.
